In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath(".."))

import requests
import pandas as pd
import re
import json
import time
from Bio import Entrez

from src.ena_fetcher import (
    search_ena_studies, 
    fetch_runs_for_study, 
    resolve_host_species,
    fetch_study_origin,
    fetch_abstract_from_pmid
)
from src.fetcher import configure_entrez

configure_entrez()
print("ready")

Entrez configured with email: akharya@ucsd.edu
ready


In [2]:
# load all MMC batch CSVs (ENA Accessions from PMIDs)
mmc_batch1 = pd.read_csv("../results/pmid-ena-results-2026-05-18-batch1.csv")
mmc_batch2 = pd.read_csv("../results/pmid-ena-results-2026-05-19-batch2.csv")
mmc_batch34 = pd.read_csv("../results/pmid-ena-results-2026-05-19-batch3.csv") 

print(f"Batch 1: {len(mmc_batch1)} papers, {(mmc_batch1['accession_count']>0).sum()} with accessions")
print(f"Batch 2: {len(mmc_batch2)} papers, {(mmc_batch2['accession_count']>0).sum()} with accessions")
print(f"Batch 3+4: {len(mmc_batch34)} papers, {(mmc_batch34['accession_count']>0).sum()} with accessions")

Batch 1: 500 papers, 202 with accessions
Batch 2: 500 papers, 223 with accessions
Batch 3+4: 846 papers, 310 with accessions


In [3]:
# extract all unique project accessions from all batches
all_mmc = pd.concat([mmc_batch1, mmc_batch2, mmc_batch34], ignore_index=True)

pubmed_accessions = []
for _, row in all_mmc[all_mmc["accessions"].notna()].iterrows():
    for acc in str(row["accessions"]).split(";"):
        acc = acc.strip()
        if re.match(r'PRJ[ENDB][A-Z]\d+', acc):
            pubmed_accessions.append(acc)

pubmed_accessions = list(set(pubmed_accessions))
print(f"Total unique project accessions from PubMed: {len(pubmed_accessions)}")

Total unique project accessions from PubMed: 891


In [4]:
# load ENA taxonomy search results (ENA route)
ena_queries = [
    'tax_tree(749906) AND library_strategy="WGS"',
    'tax_tree(1861841) AND library_strategy="WGS"',
    'tax_tree(506600) AND library_strategy="WGS"',
    'tax_tree(410661) AND library_strategy="WGS"',
    'tax_tree(1510822) AND library_strategy="WGS"',
    'tax_tree(1602388) AND library_strategy="WGS"',
    'tax_tree(408170) AND library_strategy="WGS"',
]

ena_accessions = set()
for query in ena_queries:
    accessions = search_ena_studies(query, max_results=500)
    ena_accessions.update(accessions)
    print(f"{query.split('tax_tree(')[1].split(')')[0]} → {len(accessions)} studies")
    time.sleep(1)

print(f"\nTotal ENA taxonomy accessions: {len(ena_accessions)}")

# combine both sources
all_accessions = set(pubmed_accessions) | ena_accessions
print(f"Total unique accessions combined: {len(all_accessions)}")

Found 5000 runs, 66 unique studies
749906 → 66 studies
Found 5000 runs, 204 unique studies
1861841 → 204 studies
Found 5000 runs, 89 unique studies
506600 → 89 studies
Found 5000 runs, 77 unique studies
410661 → 77 studies
Found 5000 runs, 112 unique studies
1510822 → 112 studies
Found 3160 runs, 63 unique studies
1602388 → 63 studies
Found 5000 runs, 70 unique studies
408170 → 70 studies

Total ENA taxonomy accessions: 674
Total unique accessions combined: 1502


In [5]:
import json

all_accessions_list = list(all_accessions)

with open("../results/all_accessions.json", "w") as f:
    json.dump(all_accessions_list, f)

print(f"Saved {len(all_accessions_list)} accessions to all_accessions.json")

Saved 1502 accessions to all_accessions.json


In [6]:
acc_to_pmid = {}
for _, row in all_mmc[all_mmc["accessions"].notna()].iterrows():
    pmid = str(row["pmid"])
    for acc in str(row["accessions"]).split(";"):
        acc = acc.strip()
        if re.match(r'PRJ[ENDB][A-Z]\d+', acc):
            acc_to_pmid[acc] = pmid

with open("../results/acc_to_pmid.json", "w") as f:
    json.dump(acc_to_pmid, f)

print(f"Accession to PMID mappings: {len(acc_to_pmid)}")
print(f"Studies with abstracts available: {len(acc_to_pmid)}")
print(f"Studies without abstracts (ENA only): {len(all_accessions) - len(acc_to_pmid)}")

Accession to PMID mappings: 891
Studies with abstracts available: 891
Studies without abstracts (ENA only): 611


In [7]:
# fetch metadata for all 1502 studies
results = []

for i, study in enumerate(all_accessions_list):
    if i % 50 == 0:
        print(f"Progress: {i}/{len(all_accessions_list)}")
    
    runs_df = fetch_runs_for_study(study)
    if runs_df is None or len(runs_df) == 0:
        continue
    
    results.append({
        "study_accession": study,
        "host_species": resolve_host_species(runs_df),
        "host_tax_id": runs_df["host_tax_id"].dropna().mode()[0] if len(runs_df["host_tax_id"].dropna()) > 0 else None,
        "body_site": runs_df["host_body_site"].dropna().mode()[0] if len(runs_df["host_body_site"].dropna()) > 0 else "",
        "country": runs_df["country"].dropna().mode()[0] if len(runs_df["country"].dropna()) > 0 else "",
        "n_samples": runs_df["sample_accession"].nunique(),
        "library_strategy": runs_df["library_strategy"].dropna().mode()[0] if len(runs_df["library_strategy"].dropna()) > 0 else None,
        "data_source": "pubmed" if study in acc_to_pmid else "ena_only",
        "pmid": acc_to_pmid.get(study)
    })
    time.sleep(0.3)

# save immediately
pd.DataFrame(results).to_csv("../results/catalog_raw.tsv", sep="\t", index=False)
print(f"\nFetched metadata for {len(results)} studies")
print(f"Saved to catalog_raw.tsv")

Progress: 0/1502
No runs found for PRJNA1310243
No runs found for PRJNA1416997
No runs found for PRJNA10203991


KeyboardInterrupt: 

In [8]:
# check if acc_to_pmid has the same accessions as pubmed_accessions
pubmed_set = set(pubmed_accessions)
pmid_set = set(acc_to_pmid.keys())

print(f"pubmed_accessions: {len(pubmed_set)}")
print(f"acc_to_pmid keys: {len(pmid_set)}")
print(f"Are they identical? {pubmed_set == pmid_set}")
print(f"In pubmed but not pmid mapping: {len(pubmed_set - pmid_set)}")
print(f"In pmid mapping but not pubmed: {len(pmid_set - pubmed_set)}")

pubmed_accessions: 891
acc_to_pmid keys: 891
Are they identical? True
In pubmed but not pmid mapping: 0
In pmid mapping but not pubmed: 0


In [9]:
# fetch metadata for all 1502 studies
results = []

for i, study in enumerate(all_accessions_list):
    if i % 50 == 0:
        print(f"Progress: {i}/{len(all_accessions_list)}")
    
    runs_df = fetch_runs_for_study(study)
    if runs_df is None or len(runs_df) == 0:
        continue
    
    results.append({
        "study_accession": study,
        "host_species": resolve_host_species(runs_df),
        "host_tax_id": runs_df["host_tax_id"].dropna().mode()[0] if len(runs_df["host_tax_id"].dropna()) > 0 else None,
        "body_site": runs_df["host_body_site"].dropna().mode()[0] if len(runs_df["host_body_site"].dropna()) > 0 else "",
        "country": runs_df["country"].dropna().mode()[0] if len(runs_df["country"].dropna()) > 0 else "",
        "n_samples": runs_df["sample_accession"].nunique(),
        "library_strategy": runs_df["library_strategy"].dropna().mode()[0] if len(runs_df["library_strategy"].dropna()) > 0 else None,
        "data_source": "pubmed" if study in acc_to_pmid else "ena_only",
        "pmid": acc_to_pmid.get(study)
    })
    time.sleep(0.3)

# save immediately
pd.DataFrame(results).to_csv("../results/catalog_raw.tsv", sep="\t", index=False)
print(f"\nFetched metadata for {len(results)} studies")
print(f"Saved to catalog_raw.tsv")

Progress: 0/1502
No runs found for PRJNA1310243
No runs found for PRJNA1416997
No runs found for PRJNA10203991
Progress: 50/1502
No runs found for PRJEB35945
No runs found for PRJNA95448
Progress: 100/1502
No runs found for PRJNA1205488
No runs found for PRJNA765142
Progress: 150/1502
No runs found for PRJDB4392
No runs found for PRJNA1023627
No runs found for PRJEB43619
No runs found for PRJNA925792
No runs found for PRJNA836733
No runs found for PRJNA1211720
No runs found for PRJNA20689
No runs found for PRJNA888994
Progress: 200/1502
No runs found for PRJNA657473
No runs found for PRJNA288223
No runs found for PRJNA1193846
No runs found for PRJEB14698
Progress: 250/1502
No runs found for PRJEB20626
No runs found for PRJNA1217447
No runs found for PRJNA977461
Progress: 300/1502
No runs found for PRJNA1224300
No runs found for PRJNA755318
No runs found for PRJNA647163
No runs found for PRJEB1046
No runs found for PRJNA810349
No runs found for PRJEB801559
No runs found for PRJNA6844542

In [10]:
# check what we have
catalog_raw = pd.read_csv("../results/catalog_raw.tsv", sep="\t")

print(f"Total studies: {len(catalog_raw)}")
print(f"PubMed studies: {(catalog_raw['data_source'] == 'pubmed').sum()}")
print(f"ENA-only studies: {(catalog_raw['data_source'] == 'ena_only').sum()}")
print(f"Total samples: {catalog_raw['n_samples'].sum()}")
print(f"\nTop library strategies:")
print(catalog_raw['library_strategy'].value_counts().head(10))
print(f"\nStudies with host resolved: {catalog_raw['host_species'].notna().sum()}")

Total studies: 1390
PubMed studies: 779
ENA-only studies: 611
Total samples: 172104

Top library strategies:
library_strategy
WGS                 899
AMPLICON            374
RNA-Seq              49
OTHER                40
WGA                  18
Targeted-Capture      2
RAD-Seq               2
CLONEEND              2
ChIP-Seq              2
Hi-C                  1
Name: count, dtype: int64

Studies with host resolved: 1218


In [11]:
# keep only WGS and METAGENOMIC studies
wgs_catalog = catalog_raw[catalog_raw["library_strategy"].isin(["WGS", "METAGENOMIC", "WGA"])]

print(f"WGS/Metagenomic studies: {len(wgs_catalog)}")
print(f"Total WGS samples: {wgs_catalog['n_samples'].sum()}")
print(f"Host resolved: {wgs_catalog['host_species'].notna().sum()}")

WGS/Metagenomic studies: 917
Total WGS samples: 105289
Host resolved: 775


In [12]:
wgs_catalog = catalog_raw[catalog_raw["library_strategy"].isin(["WGS", "METAGENOMIC", "WGA"])]

print(f"WGS studies: {len(wgs_catalog)}")
print(f"WGS samples: {wgs_catalog['n_samples'].sum()}")

# check host species to identify non-microbiome studies
print(f"\nTop host species in WGS studies:")
print(wgs_catalog['host_species'].value_counts().head(20))

WGS studies: 917
WGS samples: 105289

Top host species in WGS studies:
host_species
Homo sapiens                156
Mus musculus                 67
Gallus gallus                63
Sus scrofa                   62
feces metagenome             35
Sus scrofa domesticus        21
Bos taurus                   14
fish gut metagenome          13
missing                      13
pig gut metagenome           11
mouse                        10
chicken gut metagenome       10
Canis lupus familiaris        9
Salmo salar                   7
rat                           6
Felis catus                   6
Gallus gallus domesticus      6
Rattus norvegicus             5
Equus caballus                4
Escherichia coli              4
Name: count, dtype: int64


In [13]:
# identify likely metagenome studies
metagenome_terms = ["metagenome", "microbiome", "microbiota"]

def is_likely_metagenome(row):
    # if host_species contains metagenome terms - definitely microbiome
    host = str(row.get("host_species", "")).lower()
    if any(term in host for term in metagenome_terms):
        return True
    # if from pubmed and has a real animal host - likely microbiome
    if row.get("data_source") == "pubmed":
        return True
    return False

wgs_catalog["likely_metagenome"] = wgs_catalog.apply(is_likely_metagenome, axis=1)

print(f"Likely metagenome studies: {wgs_catalog['likely_metagenome'].sum()}")
print(f"Possibly non-metagenome: {(~wgs_catalog['likely_metagenome']).sum()}")
print(f"\nSamples in likely metagenome studies: {wgs_catalog[wgs_catalog['likely_metagenome']]['n_samples'].sum()}")

Likely metagenome studies: 440
Possibly non-metagenome: 477

Samples in likely metagenome studies: 51066


/var/folders/ws/dmbkn8cx75l33lp2zph5rpbw0000gn/T/ipykernel_45649/978602913.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  wgs_catalog["likely_metagenome"] = wgs_catalog.apply(is_likely_metagenome, axis=1)


In [14]:
def verify_is_metagenome(study_accession, title, abstract, client):
    """
    Use Claude to verify if a study is a genuine gut microbiome study.
    Returns True/False and a confidence level.
    """
    text = f"Title: {title or 'Not available'}\n\nAbstract: {abstract or 'Not available'}"
    
    prompt = f"""Is this a shotgun metagenomics study of a microbial community 
associated with an animal host (gut, fecal, intestinal)?

{text}

Answer with exactly one of:
YES - this is an animal gut/intestinal shotgun metagenomics study
NO - this is not a gut metagenomics study (explain briefly why)
UNCLEAR - cannot determine from available information"""

    try:
        message = client.messages.create(
            model="claude-haiku-4-5",
            max_tokens=100,
            system="You are a bioinformatics expert classifying sequencing studies.",
            messages=[{"role": "user", "content": prompt}]
        )
        result = message.content[0].text.strip()
        return result
    except Exception as e:
        return f"ERROR: {e}"

In [15]:
# fetch abstracts for all WGS studies with PMIDs
wgs_abstracts = {}

wgs_with_pmid = wgs_catalog[wgs_catalog["pmid"].notna()]
print(f"WGS studies with PMID: {len(wgs_with_pmid)}")

for i, row in wgs_with_pmid.iterrows():
    pmid = str(row["pmid"])
    if i % 50 == 0:
        print(f"Progress: {len(wgs_abstracts)}/{len(wgs_with_pmid)}")
    
    abstract = fetch_abstract_from_pmid(pmid)
    wgs_abstracts[row["study_accession"]] = abstract
    time.sleep(0.3)

# save abstracts
with open("../results/wgs_abstracts.json", "w") as f:
    json.dump(wgs_abstracts, f)

found = sum(1 for a in wgs_abstracts.values() if a)
print(f"\nAbstracts found: {found}/{len(wgs_with_pmid)}")

WGS studies with PMID: 375
Progress: 0/375
Progress: 14/375
Progress: 77/375
Progress: 91/375
Progress: 162/375
Progress: 180/375
Progress: 191/375
Progress: 315/375
Progress: 332/375

Abstracts found: 375/375


In [17]:
# add abstracts to wgs_catalog using .loc
wgs_catalog.loc[:, "abstract"] = wgs_catalog["study_accession"].map(wgs_abstracts)

# save abstracts
with open("../results/wgs_abstracts.json", "w") as f:
    json.dump(wgs_abstracts, f)

print(f"Studies with abstracts: {wgs_catalog['abstract'].notna().sum()}")
print(f"Studies without abstracts: {wgs_catalog['abstract'].isna().sum()}")

Studies with abstracts: 375
Studies without abstracts: 542


In [18]:
# fetch titles for studies that don't have them yet
titles = {}
for i, row in wgs_catalog.iterrows():
    if i % 50 == 0:
        print(f"Progress: {len(titles)}/{len(wgs_catalog)}")
    
    provenance = fetch_study_origin(row["study_accession"])
    titles[row["study_accession"]] = provenance.get("title")
    time.sleep(0.3)

wgs_catalog["title"] = wgs_catalog["study_accession"].map(titles)

# save
with open("../results/wgs_titles.json", "w") as f:
    json.dump(titles, f)

print(f"Studies with titles: {wgs_catalog['title'].notna().sum()}")

Progress: 0/917
Progress: 34/917
Progress: 67/917
Progress: 132/917
Progress: 168/917
Progress: 196/917
Progress: 229/917
Progress: 413/917
Progress: 445/917
Progress: 478/917


KeyboardInterrupt: 

In [19]:
from concurrent.futures import ThreadPoolExecutor
import threading

def fetch_title_safe(study_accession):
    try:
        provenance = fetch_study_origin(study_accession)
        time.sleep(0.1)
        return study_accession, provenance.get("title")
    except Exception as e:
        return study_accession, None

print(f"Fetching titles for {len(wgs_catalog)} studies...")

with ThreadPoolExecutor(max_workers=5) as executor:
    results = list(executor.map(
        fetch_title_safe,
        wgs_catalog["study_accession"].tolist()
    ))

titles = {acc: title for acc, title in results}
wgs_catalog.loc[:, "title"] = wgs_catalog["study_accession"].map(titles)

# save
with open("../results/wgs_titles.json", "w") as f:
    json.dump(titles, f)

print(f"Studies with titles: {wgs_catalog['title'].notna().sum()}")
print(f"Studies without titles: {wgs_catalog['title'].isna().sum()}")

Fetching titles for 917 studies...
Studies with titles: 915
Studies without titles: 2


/var/folders/ws/dmbkn8cx75l33lp2zph5rpbw0000gn/T/ipykernel_45649/4169092594.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  wgs_catalog.loc[:, "title"] = wgs_catalog["study_accession"].map(titles)


In [20]:
wgs_catalog = catalog_raw[catalog_raw["library_strategy"].isin(["WGS", "METAGENOMIC", "WGA"])].copy()
wgs_catalog["abstract"] = wgs_catalog["study_accession"].map(wgs_abstracts)
wgs_catalog["title"] = wgs_catalog["study_accession"].map(titles)

In [21]:
wgs_catalog.to_csv("../results/wgs_catalog.tsv", sep="\t", index=False)
print(f"Saved {len(wgs_catalog)} WGS studies to wgs_catalog.tsv")
print(f"Columns: {list(wgs_catalog.columns)}")

Saved 917 WGS studies to wgs_catalog.tsv
Columns: ['study_accession', 'host_species', 'host_tax_id', 'body_site', 'country', 'n_samples', 'library_strategy', 'data_source', 'pmid', 'abstract', 'title']


In [24]:
import anthropic
from concurrent.futures import ThreadPoolExecutor

client = anthropic.Anthropic()

def verify_is_metagenome(row):
    study_accession = row["study_accession"]
    title = row.get("title") if isinstance(row.get("title"), str) else "Not available"
    abstract = row.get("abstract") if isinstance(row.get("abstract"), str) else "Not available"
    abstract_text = abstract if isinstance(abstract, str) else "Not available"
    
    prompt = f"""Is this a shotgun metagenomics study of a microbial community associated with an animal host (gut, fecal, intestinal)?

Title: {title}
Abstract: {abstract[:500] if abstract != "Not available" else "Not available"}

Answer with exactly one of:
YES - this is an animal gut/intestinal shotgun metagenomics study
NO - this is not a gut metagenomics study (explain briefly why)
UNCLEAR - cannot determine from available information"""

    try:
        message = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=10,
            system="You are a bioinformatics expert classifying sequencing studies.",
            messages=[{"role": "user", "content": prompt}]
        )
        return study_accession, message.content[0].text.strip()
    except Exception as e:
        return study_accession, f"ERROR: {e}"

print(f"Verifying {len(wgs_catalog)} studies...")

rows = wgs_catalog.to_dict("records")

with ThreadPoolExecutor(max_workers=20) as executor:
    results = list(executor.map(verify_is_metagenome, rows))

verification = {acc: verdict for acc, verdict in results}

# add to catalog
wgs_catalog = wgs_catalog.copy()
wgs_catalog["claude_verification"] = wgs_catalog["study_accession"].map(verification)

# save
wgs_catalog.to_csv("../results/wgs_catalog_verified.tsv", sep="\t", index=False)

# summary
yes = sum(1 for v in verification.values() if v.startswith("YES"))
no = sum(1 for v in verification.values() if v.startswith("NO"))
unclear = sum(1 for v in verification.values() if v.startswith("UNCLEAR"))
print(f"\nYES: {yes}")
print(f"NO: {no}")
print(f"UNCLEAR: {unclear}")

Verifying 917 studies...


KeyboardInterrupt: 

In [25]:
import re

def classify_study(row):
    title = str(row.get("title") or "").lower()
    abstract = str(row.get("abstract") or "").lower()
    text = title + " " + abstract
    
    # definite NO terms
    no_terms = [
        "genome sequence", "whole genome sequencing", "draft genome",
        "chromosome", "transcriptome", "rna-seq", "gene expression",
        "genome assembly", "reference genome", "resequencing",
        "snp", "qtl", "gwas", "population genetics"
    ]
    
    # definite YES terms
    yes_terms = [
        "microbiome", "microbiota", "metagenome", "metagenomics",
        "gut bacteria", "gut flora", "intestinal bacteria", "fecal bacteria",
        "16s", "shotgun metagenomics", "whole metagenome"
    ]
    
    if any(term in text for term in no_terms):
        return "NO_regex"
    if any(term in text for term in yes_terms):
        return "YES_regex"
    return "UNCLEAR_needs_claude"

wgs_catalog["classification"] = wgs_catalog.apply(classify_study, axis=1)

print(wgs_catalog["classification"].value_counts())

classification
YES_regex               663
UNCLEAR_needs_claude    209
NO_regex                 45
Name: count, dtype: int64


In [26]:
# save confirmed studies
confirmed = wgs_catalog[wgs_catalog["classification"] == "YES_regex"].copy()
confirmed.to_csv("../results/confirmed_metagenome.tsv", sep="\t", index=False)
print(f"Confirmed metagenome studies: {len(confirmed)}")
print(f"Total samples: {confirmed['n_samples'].sum()}")

Confirmed metagenome studies: 663
Total samples: 74220


In [27]:
# add classification method
wgs_catalog["classification_method"] = wgs_catalog["classification"].map({
    "YES_regex": "regex",
    "NO_regex": "regex", 
    "UNCLEAR_needs_claude": "pending_claude"
})

In [28]:
unclear = wgs_catalog[wgs_catalog["classification"] == "UNCLEAR_needs_claude"].copy()
unclear_rows = unclear.to_dict("records")

counter = [0]
lock = threading.Lock()

def verify_unclear(row):
    with lock:
        counter[0] += 1
        if counter[0] % 20 == 0:
            print(f"Progress: {counter[0]}/209")
    
    study_accession = row["study_accession"]
    title = row.get("title") if isinstance(row.get("title"), str) else "Not available"
    abstract = row.get("abstract") if isinstance(row.get("abstract"), str) else "Not available"
    
    prompt = f"""Title: {title}
Abstract: {abstract[:300] if isinstance(abstract, str) else "Not available"}

Is this a shotgun metagenomics study of animal gut/intestinal microbiome?
Answer YES, NO, or UNCLEAR only."""

    try:
        message = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=10,
            system="Answer YES, NO, or UNCLEAR only.",
            messages=[{"role": "user", "content": prompt}]
        )
        return study_accession, message.content[0].text.strip()
    except Exception as e:
        return study_accession, "ERROR"

with ThreadPoolExecutor(max_workers=10) as executor:
    results = list(executor.map(verify_unclear, unclear_rows))

unclear_verification = {acc: verdict for acc, verdict in results}
print(f"\nDone. Results:")
print(pd.Series(list(unclear_verification.values())).value_counts())

KeyboardInterrupt: 

In [29]:
import pandas as pd

# load postdoc's 652 studies
postdoc_df = pd.read_csv("../data/animal_gut_shotgun_studies - Studies.csv")
print(f"Postdoc studies: {len(postdoc_df)}")
print(f"Columns: {list(postdoc_df.columns)}")

# check how many have ENA accessions already
has_accessions = postdoc_df["Accessions (extracted)"].notna() & (postdoc_df["Accessions (extracted)"] != "")
print(f"Studies with accessions already: {has_accessions.sum()}")

Postdoc studies: 652
Columns: ['PMID', 'PubMed Link', 'PMCID', 'PMC Link', 'DOI', 'DOI Link', 'Year', 'Journal', 'Title', 'First Author', 'Last Author', 'Primary Class', 'Host (common, detected)', 'Host Class(es) detected', 'Binomials (heuristic)', 'Accessions (extracted)', 'PubTypes', 'Abstract', 'MeSH terms', 'Reviewed?', 'Include?', 'Notes (manual)']
Studies with accessions already: 18


In [30]:
# save PMIDs for Alan's tool
pmids = postdoc_df["PMID"].dropna().astype(int).astype(str).tolist()
print(f"Total PMIDs: {len(pmids)}")

with open("../results/postdoc_pmids.txt", "w") as f:
    for pmid in pmids:
        f.write(pmid + "\n")

print("Saved to postdoc_pmids.txt")

Total PMIDs: 652
Saved to postdoc_pmids.txt


In [31]:
# check overlap with your earlier PubMed search
your_pmids = set(all_mmc["pmid"].astype(str).tolist())
postdoc_pmids = set(postdoc_df["PMID"].dropna().astype(int).astype(str).tolist())

overlap = your_pmids & postdoc_pmids
print(f"Your PubMed search: {len(your_pmids)} papers")
print(f"Postdoc's search: {len(postdoc_pmids)} papers")
print(f"Overlap: {len(overlap)} papers in both")
print(f"New in postdoc's search: {len(postdoc_pmids - your_pmids)} papers")

Your PubMed search: 1843 papers
Postdoc's search: 652 papers
Overlap: 325 papers in both
New in postdoc's search: 327 papers


In [32]:
import re

# load Alan's results for postdoc's papers
mmc_postdoc = pd.read_csv("../results/pmid-ena-results-2026-05-26(sam's).csv")  

# extract all project accessions
postdoc_accessions = []
for _, row in mmc_postdoc[mmc_postdoc["accessions"].notna()].iterrows():
    for acc in str(row["accessions"]).split(";"):
        acc = acc.strip()
        if re.match(r'PRJ[ENDB][A-Z]\d+', acc):
            postdoc_accessions.append(acc)

postdoc_accessions = list(set(postdoc_accessions))
print(f"Total project accessions from postdoc search: {len(postdoc_accessions)}")

# find new ones not already in your catalog
existing = set(all_accessions_list)
new_from_postdoc = [a for a in postdoc_accessions if a not in existing]
print(f"Already in catalog: {len(postdoc_accessions) - len(new_from_postdoc)}")
print(f"New accessions to add: {len(new_from_postdoc)}")

Total project accessions from postdoc search: 399
Already in catalog: 230
New accessions to add: 169


In [33]:
new_results_postdoc = []

for i, study in enumerate(new_from_postdoc):
    if i % 20 == 0:
        print(f"Progress: {i}/{len(new_from_postdoc)}")
    
    runs_df = fetch_runs_for_study(study)
    if runs_df is None or len(runs_df) == 0:
        continue
    
    new_results_postdoc.append({
        "study_accession": study,
        "host_species": resolve_host_species(runs_df),
        "host_tax_id": runs_df["host_tax_id"].dropna().mode()[0] if len(runs_df["host_tax_id"].dropna()) > 0 else None,
        "body_site": runs_df["host_body_site"].dropna().mode()[0] if len(runs_df["host_body_site"].dropna()) > 0 else "",
        "country": runs_df["country"].dropna().mode()[0] if len(runs_df["country"].dropna()) > 0 else "",
        "n_samples": runs_df["sample_accession"].nunique(),
        "library_strategy": runs_df["library_strategy"].dropna().mode()[0] if len(runs_df["library_strategy"].dropna()) > 0 else None,
        "data_source": "pubmed_postdoc",
        "pmid": mmc_postdoc[mmc_postdoc["accessions"].str.contains(study, na=False)]["pmid"].iloc[0] if len(mmc_postdoc[mmc_postdoc["accessions"].str.contains(study, na=False)]) > 0 else None
    })
    time.sleep(0.3)

print(f"\nFetched {len(new_results_postdoc)} new studies")

Progress: 0/169
No runs found for PRJNA1358044
Progress: 20/169
No runs found for PRJNA864640
No runs found for PRJNA261082
Progress: 40/169
No runs found for PRJNA1277365
Progress: 60/169
No runs found for PRJNA648267
No runs found for PRJNA983434
No runs found for PRJNA546576
No runs found for PRJNA1250387
No runs found for PRJNA1010509
Progress: 80/169
No runs found for PRJNA1247262
No runs found for PRJNA1024450
No runs found for PRJNA556087
Progress: 100/169
No runs found for PRJNA1024074
No runs found for PRJNA1022859
No runs found for PRJNA1010808
No runs found for PRJNA1038784
Progress: 120/169
No runs found for PRJNA1185452
No runs found for PRJNA1185451
No runs found for PRJNA662678
No runs found for PRJDB7102
No runs found for PRJNA1085270
No runs found for PRJNA1024099
No runs found for PRJNA1052115
Progress: 140/169
Progress: 160/169
No runs found for PRJEB70999

Fetched 145 new studies


In [34]:
# load existing catalog
catalog_raw = pd.read_csv("../results/catalog_raw.tsv", sep="\t")

# add new studies from postdoc search
new_df = pd.DataFrame(new_results_postdoc)

# combine and deduplicate
final_catalog = pd.concat([catalog_raw, new_df], ignore_index=True)
final_catalog = final_catalog.drop_duplicates(subset=["study_accession"])

print(f"Previous catalog: {len(catalog_raw)}")
print(f"New studies added: {len(new_df)}")
print(f"Final catalog: {len(final_catalog)}")
print(f"Total samples: {final_catalog['n_samples'].sum()}")

Previous catalog: 1390
New studies added: 145
Final catalog: 1535
Total samples: 185322


In [35]:
# merge postdoc's rich metadata into your catalog
postdoc_clean = postdoc_df[[
    "PMID", "Title", "First Author", "Last Author", 
    "Primary Class", "Abstract", "Year", "Journal"
]].rename(columns={
    "PMID": "pmid_str",
    "Title": "paper_title",
    "First Author": "first_author",
    "Last Author": "last_author",
    "Primary Class": "host_class_postdoc",
    "Abstract": "abstract_postdoc",
    "Year": "pub_year",
    "Journal": "journal"
})

# convert pmid to string for matching
postdoc_clean["pmid_str"] = postdoc_clean["pmid_str"].astype(str)
final_catalog["pmid_str"] = final_catalog["pmid"].astype(str)

# merge
final_catalog = final_catalog.merge(postdoc_clean, on="pmid_str", how="left")

print(f"Studies with postdoc metadata: {final_catalog['host_class_postdoc'].notna().sum()}")
print(f"Studies with abstract: {final_catalog['abstract_postdoc'].notna().sum()}")

# save
final_catalog.to_csv("../results/final_catalog.tsv", sep="\t", index=False)
print(f"\nSaved {len(final_catalog)} studies to final_catalog.tsv")

Studies with postdoc metadata: 0
Studies with abstract: 0

Saved 1535 studies to final_catalog.tsv


In [36]:
# check what pmid looks like in both dataframes
print("Your catalog pmid sample:")
print(final_catalog["pmid"].dropna().head(5).tolist())
print(type(final_catalog["pmid"].dropna().iloc[0]))

print("\nPostdoc pmid sample:")
print(postdoc_df["PMID"].dropna().head(5).tolist())
print(type(postdoc_df["PMID"].dropna().iloc[0]))

Your catalog pmid sample:
[28951889.0, 31959971.0, 33824199.0, 32600361.0, 32723789.0]
<class 'numpy.float64'>

Postdoc pmid sample:
[42083207, 42066496, 42059394, 42039826, 41994282]
<class 'numpy.int64'>


In [37]:
# fix pmid format in both dataframes
final_catalog["pmid_str"] = final_catalog["pmid"].dropna().apply(lambda x: str(int(x))).reindex(final_catalog.index)
postdoc_clean["pmid_str"] = postdoc_df["PMID"].astype(str)

# merge again
final_catalog = final_catalog.drop(columns=["pmid_str"], errors="ignore")
final_catalog["pmid_str"] = final_catalog["pmid"].apply(lambda x: str(int(x)) if pd.notna(x) else None)

final_catalog = final_catalog.merge(postdoc_clean, on="pmid_str", how="left")

print(f"Studies with postdoc metadata: {final_catalog['host_class_postdoc'].notna().sum()}")
print(f"Studies with abstract: {final_catalog['abstract_postdoc'].notna().sum()}")

final_catalog.to_csv("../results/final_catalog.tsv", sep="\t", index=False)
print(f"Saved {len(final_catalog)} studies")

KeyError: 'host_class_postdoc'

In [38]:
print(final_catalog.columns.tolist())

['study_accession', 'host_species', 'host_tax_id', 'body_site', 'country', 'n_samples', 'library_strategy', 'data_source', 'pmid', 'paper_title_x', 'first_author_x', 'last_author_x', 'host_class_postdoc_x', 'abstract_postdoc_x', 'pub_year_x', 'journal_x', 'pmid_str', 'paper_title_y', 'first_author_y', 'last_author_y', 'host_class_postdoc_y', 'abstract_postdoc_y', 'pub_year_y', 'journal_y']


In [39]:
# drop duplicate columns and keep the good ones
final_catalog = final_catalog.drop(columns=[
    "paper_title_x", "first_author_x", "last_author_x",
    "host_class_postdoc_x", "abstract_postdoc_x", "pub_year_x", "journal_x"
])

# rename _y columns to clean names
final_catalog = final_catalog.rename(columns={
    "paper_title_y": "paper_title",
    "first_author_y": "first_author",
    "last_author_y": "last_author",
    "host_class_postdoc_y": "host_class",
    "abstract_postdoc_y": "abstract",
    "pub_year_y": "pub_year",
    "journal_y": "journal"
})

print(f"Studies with host class: {final_catalog['host_class'].notna().sum()}")
print(f"Studies with abstract: {final_catalog['abstract'].notna().sum()}")
print(f"Studies with title: {final_catalog['paper_title'].notna().sum()}")

final_catalog.to_csv("../results/final_catalog.tsv", sep="\t", index=False)
print(f"\nSaved {len(final_catalog)} studies")

Studies with host class: 310
Studies with abstract: 321
Studies with title: 321

Saved 1535 studies


In [40]:
# final summary
print(f"Total studies: {len(final_catalog)}")
print(f"Total samples: {final_catalog['n_samples'].sum()}")
print(f"With abstract: {final_catalog['abstract'].notna().sum()}")
print(f"With host class: {final_catalog['host_class'].notna().sum()}")
print(f"With host species: {final_catalog['host_species'].notna().sum()}")
print(f"\nData sources:")
print(final_catalog['data_source'].value_counts())

Total studies: 1535
Total samples: 185322
With abstract: 321
With host class: 310
With host species: 1352

Data sources:
data_source
pubmed            779
ena_only          611
pubmed_postdoc    145
Name: count, dtype: int64


In [41]:
def classify_study(row):
    title = str(row.get("paper_title") or row.get("title") or "").lower()
    abstract = str(row.get("abstract") or "").lower()
    text = title + " " + abstract
    
    no_terms = [
        "genome sequence", "whole genome sequencing", "draft genome",
        "chromosome", "transcriptome", "rna-seq", "gene expression",
        "genome assembly", "reference genome", "resequencing",
        "snp", "qtl", "gwas", "population genetics"
    ]
    
    yes_terms = [
        "microbiome", "microbiota", "metagenome", "metagenomics",
        "gut bacteria", "gut flora", "intestinal bacteria", "fecal bacteria",
        "16s", "shotgun metagenomics", "whole metagenome"
    ]
    
    if any(term in text for term in no_terms):
        return "NO"
    if any(term in text for term in yes_terms):
        return "YES"
    return "UNCLEAR"

final_catalog["classification"] = final_catalog.apply(classify_study, axis=1)

print(final_catalog["classification"].value_counts())
print(f"\nYES samples: {final_catalog[final_catalog['classification']=='YES']['n_samples'].sum()}")
print(f"Total samples if keep YES+UNCLEAR: {final_catalog[final_catalog['classification']!='NO']['n_samples'].sum()}")

classification
UNCLEAR    1215
YES         266
NO           54
Name: count, dtype: int64

YES samples: 26364
Total samples if keep YES+UNCLEAR: 179287


In [42]:
unclear = final_catalog[final_catalog["classification"] == "UNCLEAR"]

print(f"UNCLEAR studies: {len(unclear)}")
print(f"\nData sources of UNCLEAR:")
print(unclear["data_source"].value_counts())

print(f"\nSample titles of UNCLEAR:")
print(unclear["paper_title"].dropna().head(10).tolist())

print(f"\nHost species in UNCLEAR:")
print(unclear["host_species"].value_counts().head(15))

UNCLEAR studies: 1215

Data sources of UNCLEAR:
data_source
ena_only          611
pubmed            603
pubmed_postdoc      1
Name: count, dtype: int64

Sample titles of UNCLEAR:
['Characterization and Preliminary Application of a Novel Lytic Vibrio parahaemolyticus Bacteriophage vB_VpaP_SJSY21.']

Host species in UNCLEAR:
host_species
Homo sapiens              193
Mus musculus              133
Sus scrofa                 78
Gallus gallus              75
feces metagenome           37
Sus scrofa domesticus      28
mouse                      24
Bos taurus                 23
Canis lupus familiaris     17
fish gut metagenome        15
pig gut metagenome         14
missing                    14
chicken gut metagenome     13
Rattus norvegicus          12
Ovis aries                  7
Name: count, dtype: int64


In [43]:
unclear_pubmed = unclear[unclear["data_source"].isin(["pubmed", "pubmed_postdoc"])]

# show studies that have abstracts but are still UNCLEAR
has_abstract = unclear_pubmed[unclear_pubmed["abstract"].notna()]
print(f"UNCLEAR pubmed studies WITH abstracts: {len(has_abstract)}")
print(f"UNCLEAR pubmed studies WITHOUT abstracts: {len(unclear_pubmed) - len(has_abstract)}")

# show a few abstracts to understand why they're unclear
for i, row in has_abstract.head(3).iterrows():
    print(f"\n--- {row['study_accession']} ---")
    print(f"Title: {row.get('paper_title', 'None')}")
    print(f"Abstract snippet: {str(row['abstract'])[:300]}")

UNCLEAR pubmed studies WITH abstracts: 1
UNCLEAR pubmed studies WITHOUT abstracts: 603

--- PRJNA1025901 ---
Title: Characterization and Preliminary Application of a Novel Lytic Vibrio parahaemolyticus Bacteriophage vB_VpaP_SJSY21.
Abstract snippet: Litopenaeus vannamei is one of the most economically significant aquatic species globally. However, the emergence of acute hepatopancreatic necrosis disease (AHPND) in recent years has resulted in substantial losses within the L. vannamei farming industry. Phage therapy holds promise as an effective


In [44]:
# get pmids for unclear pubmed studies without abstracts
unclear_no_abstract = unclear_pubmed[unclear_pubmed["abstract"].isna()]
pmids_to_fetch = unclear_no_abstract["pmid"].dropna().apply(
    lambda x: str(int(x)) if pd.notna(x) else None
).dropna().tolist()

print(f"PMIDs to fetch abstracts for: {len(pmids_to_fetch)}")

PMIDs to fetch abstracts for: 603


In [45]:
from concurrent.futures import ThreadPoolExecutor

def fetch_abstract_safe(pmid):
    try:
        abstract = fetch_abstract_from_pmid(pmid)
        time.sleep(0.1)
        return pmid, abstract
    except Exception as e:
        return pmid, None

print("Fetching abstracts for 603 UNCLEAR PubMed studies...")

with ThreadPoolExecutor(max_workers=10) as executor:
    results = list(executor.map(fetch_abstract_safe, pmids_to_fetch))

new_abstracts = {pmid: abstract for pmid, abstract in results}
found = sum(1 for a in new_abstracts.values() if a)
print(f"Abstracts found: {found}/603")

Fetching abstracts for 603 UNCLEAR PubMed studies...
No abstract found for PubMed ID 29511208: HTTP Error 429: Too Many Requests
No abstract found for PubMed ID 42019469: HTTP Error 429: Too Many Requests
No abstract found for PubMed ID 33912174: HTTP Error 429: Too Many Requests
No abstract found for PubMed ID 36389176: HTTP Error 429: Too Many Requests
No abstract found for PubMed ID 36759753: HTTP Error 429: Too Many Requests
No abstract found for PubMed ID 35440042: HTTP Error 429: Too Many Requests
No abstract found for PubMed ID 41928361: HTTP Error 429: Too Many Requests
No abstract found for PubMed ID 38882494: HTTP Error 429: Too Many Requests
No abstract found for PubMed ID 38349151: HTTP Error 429: Too Many Requests
No abstract found for PubMed ID 35002754: HTTP Error 429: Too Many Requests
No abstract found for PubMed ID 39269772: HTTP Error 429: Too Many Requests
No abstract found for PubMed ID 40671160: HTTP Error 429: Too Many Requests
No abstract found for PubMed ID 384

KeyboardInterrupt: 

In [3]:
import sys
import os
sys.path.insert(0, os.path.abspath(".."))

import json
import pandas as pd
import requests
import time
import re
from concurrent.futures import ThreadPoolExecutor

from src.ena_fetcher import (
    search_ena_studies,
    fetch_runs_for_study,
    resolve_host_species,
    fetch_study_origin,
    fetch_abstract_from_pmid
)
from src.fetcher import configure_entrez

configure_entrez()

# reload catalog
final_catalog = pd.read_csv("../results/final_catalog.tsv", sep="\t")

# reload accessions
with open("../results/all_accessions.json") as f:
    all_accessions = json.load(f)

# reload pmid mapping
with open("../results/acc_to_pmid.json") as f:
    acc_to_pmid = json.load(f)

# reload MMC batch files
mmc_batch1 = pd.read_csv("../results/pmid-ena-results-2026-05-18-batch1.csv")
mmc_batch2 = pd.read_csv("../results/pmid-ena-results-2026-05-19-batch2.csv")
mmc_batch34 = pd.read_csv("../results/pmid-ena-results-2026-05-19-batch3.csv")
mmc_postdoc = pd.read_csv("../results/pmid-ena-results-2026-05-26(sam's).csv")
all_mmc = pd.concat([mmc_batch1, mmc_batch2, mmc_batch34, mmc_postdoc], ignore_index=True)

print(f"Catalog: {len(final_catalog)} studies")
print(f"Total samples: {final_catalog['n_samples'].sum()}")
print(f"All accessions: {len(all_accessions)}")
print(f"PMID mappings: {len(acc_to_pmid)}")

Entrez configured with email: akharya@ucsd.edu
Catalog: 1535 studies
Total samples: 185322
All accessions: 1502
PMID mappings: 891


In [4]:
# get pmids for unclear pubmed studies without abstracts
def classify_study(row):
    title = str(row.get("paper_title") or "").lower()
    abstract = str(row.get("abstract") or "").lower()
    text = title + " " + abstract
    
    no_terms = [
        "genome sequence", "whole genome sequencing", "draft genome",
        "chromosome", "transcriptome", "rna-seq", "gene expression",
        "genome assembly", "reference genome", "resequencing",
        "snp", "qtl", "gwas", "population genetics"
    ]
    
    yes_terms = [
        "microbiome", "microbiota", "metagenome", "metagenomics",
        "gut bacteria", "gut flora", "intestinal bacteria", "fecal bacteria",
        "16s", "shotgun metagenomics", "whole metagenome"
    ]
    
    if any(term in text for term in no_terms):
        return "NO"
    if any(term in text for term in yes_terms):
        return "YES"
    return "UNCLEAR"

final_catalog["classification"] = final_catalog.apply(classify_study, axis=1)

# get unclear pubmed studies without abstracts
unclear = final_catalog[final_catalog["classification"] == "UNCLEAR"]
unclear_pubmed = unclear[unclear["data_source"].isin(["pubmed", "pubmed_postdoc"])]
unclear_no_abstract = unclear_pubmed[unclear_pubmed["abstract"].isna()]

pmids_to_fetch = unclear_no_abstract["pmid"].dropna().apply(
    lambda x: str(int(x)) if pd.notna(x) else None
).dropna().tolist()

print(f"PMIDs to fetch: {len(pmids_to_fetch)}")

PMIDs to fetch: 603


In [5]:
def fetch_abstract_safe(pmid):
    try:
        abstract = fetch_abstract_from_pmid(pmid)
        time.sleep(0.15)
        return pmid, abstract
    except Exception as e:
        return pmid, None

print("Fetching abstracts for 603 studies...")

with ThreadPoolExecutor(max_workers=3) as executor:
    results = list(executor.map(fetch_abstract_safe, pmids_to_fetch))

new_abstracts = {pmid: abstract for pmid, abstract in results}
found = sum(1 for a in new_abstracts.values() if a)
print(f"Abstracts found: {found}/603")

# save immediately
import json
with open("../results/new_abstracts.json", "w") as f:
    json.dump(new_abstracts, f)
print("Saved new_abstracts.json")

Fetching abstracts for 603 studies...
Abstracts found: 430/603
Saved new_abstracts.json


In [6]:
# add new abstracts to catalog
final_catalog.loc[:, "abstract"] = final_catalog.apply(
    lambda row: new_abstracts.get(str(int(row["pmid"])) if pd.notna(row["pmid"]) else None, row.get("abstract")),
    axis=1
)

# rerun classification with updated abstracts
final_catalog["classification"] = final_catalog.apply(classify_study, axis=1)

print(final_catalog["classification"].value_counts())
print(f"\nYES samples: {final_catalog[final_catalog['classification']=='YES']['n_samples'].sum()}")
print(f"NO samples: {final_catalog[final_catalog['classification']=='NO']['n_samples'].sum()}")
print(f"UNCLEAR samples: {final_catalog[final_catalog['classification']=='UNCLEAR']['n_samples'].sum()}")

classification
YES        802
UNCLEAR    614
NO         119
Name: count, dtype: int64

YES samples: 94498
NO samples: 10994
UNCLEAR samples: 79830


In [7]:
print("=== CATALOG SUMMARY ===")
print(f"\nTotal studies: {len(final_catalog)}")
print(f"Total samples: {final_catalog['n_samples'].sum():,}")

print(f"\n--- Abstracts ---")
print(f"Studies with abstract: {final_catalog['abstract'].notna().sum()}")
print(f"Studies without abstract: {final_catalog['abstract'].isna().sum()}")

print(f"\n--- Data sources ---")
print(final_catalog['data_source'].value_counts())

print(f"\n--- Classification (regex) ---")
print(final_catalog['classification'].value_counts())
print(f"\nYES studies: {(final_catalog['classification']=='YES').sum()}")
print(f"YES samples: {final_catalog[final_catalog['classification']=='YES']['n_samples'].sum():,}")
print(f"\nNO studies: {(final_catalog['classification']=='NO').sum()}")
print(f"NO samples: {final_catalog[final_catalog['classification']=='NO']['n_samples'].sum():,}")
print(f"\nUNCLEAR studies: {(final_catalog['classification']=='UNCLEAR').sum()}")
print(f"UNCLEAR samples: {final_catalog[final_catalog['classification']=='UNCLEAR']['n_samples'].sum():,}")

print(f"\n--- Host resolution ---")
print(f"Host species resolved: {final_catalog['host_species'].notna().sum()}")
print(f"Host class available: {final_catalog['host_class'].notna().sum()}")

=== CATALOG SUMMARY ===

Total studies: 1535
Total samples: 185,322

--- Abstracts ---
Studies with abstract: 924
Studies without abstract: 611

--- Data sources ---
data_source
pubmed            779
ena_only          611
pubmed_postdoc    145
Name: count, dtype: int64

--- Classification (regex) ---
classification
YES        802
UNCLEAR    614
NO         119
Name: count, dtype: int64

YES studies: 802
YES samples: 94,498

NO studies: 119
NO samples: 10,994

UNCLEAR studies: 614
UNCLEAR samples: 79,830

--- Host resolution ---
Host species resolved: 1352
Host class available: 310


In [8]:
# flag as NO but keep in catalog
final_catalog.loc[final_catalog["classification"] == "NO", "needs_review"] = True

In [10]:
print(final_catalog.columns.tolist())

['study_accession', 'host_species', 'host_tax_id', 'body_site', 'country', 'n_samples', 'library_strategy', 'data_source', 'pmid', 'pmid_str', 'paper_title', 'first_author', 'last_author', 'host_class', 'abstract', 'pub_year', 'journal', 'classification', 'needs_review']


In [11]:
runs_df = fetch_runs_for_study("PRJNA836960")
print(runs_df.columns.tolist())

['run_accession', 'study_accession', 'sample_accession', 'scientific_name', 'host', 'library_strategy', 'fastq_ftp', 'host_scientific_name', 'host_tax_id', 'host_body_site', 'disease', 'country', 'lat', 'lon', 'collection_date']


In [12]:
import importlib
import src.ena_fetcher
importlib.reload(src.ena_fetcher)
from src.ena_fetcher import fetch_runs_for_study

runs_df = fetch_runs_for_study("PRJNA836960")
print(runs_df["library_source"].value_counts())

library_source
METAGENOMIC    130
Name: count, dtype: int64


In [13]:
from concurrent.futures import ThreadPoolExecutor

def fetch_library_source(study_accession):
    try:
        runs_df = fetch_runs_for_study(study_accession)
        if runs_df is None or len(runs_df) == 0:
            return study_accession, None
        if "library_source" in runs_df.columns:
            source = runs_df["library_source"].dropna().mode()
            return study_accession, source[0] if len(source) > 0 else None
        return study_accession, None
    except Exception as e:
        return study_accession, None

print(f"Fetching library_source for {len(final_catalog)} studies...")

with ThreadPoolExecutor(max_workers=5) as executor:
    results = list(executor.map(
        fetch_library_source,
        final_catalog["study_accession"].tolist()
    ))

library_sources = {acc: src for acc, src in results}
final_catalog["library_source"] = final_catalog["study_accession"].map(library_sources)

print(final_catalog["library_source"].value_counts())

Fetching library_source for 1535 studies...
library_source
METAGENOMIC                   1272
GENOMIC                        202
TRANSCRIPTOMIC                  36
OTHER                           16
METATRANSCRIPTOMIC               4
VIRAL RNA                        2
TRANSCRIPTOMIC SINGLE CELL       1
SYNTHETIC                        1
GENOMIC SINGLE CELL              1
Name: count, dtype: int64


In [15]:
def classify_final(row):
    lib_source = str(row.get("library_source") or "")
    lib_strategy = str(row.get("library_strategy") or "")
    regex_class = str(row.get("classification") or "")
    
    # definite NO by library_source
    if lib_source in ["GENOMIC", "TRANSCRIPTOMIC", "VIRAL RNA",
                      "TRANSCRIPTOMIC SINGLE CELL", "GENOMIC SINGLE CELL", "SYNTHETIC"]:
        return "NO"
    
    # definite NO by library_strategy regardless of source
    if lib_strategy in ["AMPLICON", "RNA-Seq", "ChIP-Seq",
                        "Hi-C", "RAD-Seq", "Targeted-Capture", "CLONEEND"]:
        return "NO"
    
    # strong YES - both source AND strategy confirm metagenomics
    if lib_source == "METAGENOMIC" and lib_strategy in ["WGS", "METAGENOMIC"]:
        return "YES"
    
    # weaker signal - source says metagenomics but strategy is ambiguous
    if lib_source == "METAGENOMIC" and lib_strategy == "OTHER":
        return "MAYBE"
    
    # METATRANSCRIPTOMIC
    if lib_source == "METATRANSCRIPTOMIC":
        return "MAYBE"
    
    # fall back to regex for everything else
    return regex_class

final_catalog["classification_final"] = final_catalog.apply(classify_final, axis=1)

print(final_catalog["classification_final"].value_counts())
print(f"\nYES studies: {(final_catalog['classification_final']=='YES').sum()}")
print(f"YES samples: {final_catalog[final_catalog['classification_final']=='YES']['n_samples'].sum():,}")
print(f"NO studies: {(final_catalog['classification_final']=='NO').sum()}")
print(f"MAYBE studies: {(final_catalog['classification_final']=='MAYBE').sum()}")
print(f"UNCLEAR studies: {(final_catalog['classification_final']=='UNCLEAR').sum()}")

classification_final
YES        864
NO         631
MAYBE       37
UNCLEAR      3
Name: count, dtype: int64

YES studies: 864
YES samples: 95,765
NO studies: 631
MAYBE studies: 37
UNCLEAR studies: 3


In [16]:
no_studies = final_catalog[final_catalog["classification_final"] == "NO"]

print("NO studies breakdown:")
print(f"\nBy library_source:")
print(no_studies["library_source"].value_counts())
print(f"\nBy library_strategy:")
print(no_studies["library_strategy"].value_counts())
print(f"\nSample titles of NO studies:")
for title in no_studies["paper_title"].dropna().head(5).tolist():
    print(f"  - {title}")

NO studies breakdown:

By library_source:
library_source
METAGENOMIC                   374
GENOMIC                       202
TRANSCRIPTOMIC                 36
OTHER                          12
METATRANSCRIPTOMIC              2
VIRAL RNA                       2
TRANSCRIPTOMIC SINGLE CELL      1
SYNTHETIC                       1
GENOMIC SINGLE CELL             1
Name: count, dtype: int64

By library_strategy:
library_strategy
AMPLICON            420
WGS                 140
RNA-Seq              56
OTHER                 4
Targeted-Capture      2
RAD-Seq               2
CLONEEND              2
ChIP-Seq              2
Hi-C                  1
WGA                   1
FINISHING             1
Name: count, dtype: int64

Sample titles of NO studies:
  - Genomes of Gut Bacteria from Nasonia Wasps Shed Light on Phylosymbiosis and Microbe-Assisted Hybrid Breakdown.
  - Detection and characterization of Campylobacter in air samples from poultry houses using shot-gun metagenomics - a pilot study.
  - M

In [17]:
def classify_final(row):
    lib_source = str(row.get("library_source") or "")
    lib_strategy = str(row.get("library_strategy") or "")
    regex_class = str(row.get("classification") or "")
    
    # library_source is the STRONGER signal - check it first
    
    # definite NO by library_source
    if lib_source in ["GENOMIC", "TRANSCRIPTOMIC", "VIRAL RNA",
                      "TRANSCRIPTOMIC SINGLE CELL", "GENOMIC SINGLE CELL", "SYNTHETIC"]:
        return "NO"
    
    # strong YES - source confirms metagenomics + strategy confirms WGS
    if lib_source == "METAGENOMIC" and lib_strategy in ["WGS", "METAGENOMIC"]:
        return "YES"
    
    # source confirms metagenomics but strategy is ambiguous
    if lib_source == "METAGENOMIC" and lib_strategy in ["OTHER", "AMPLICON"]:
        return "MAYBE"
    
    # source is METAGENOMIC with any other strategy - still likely YES
    if lib_source == "METAGENOMIC":
        return "YES"
    
    # METATRANSCRIPTOMIC
    if lib_source == "METATRANSCRIPTOMIC":
        return "MAYBE"
    
    # now check library_strategy for remaining studies
    if lib_strategy in ["RNA-Seq", "ChIP-Seq", "Hi-C", 
                        "RAD-Seq", "Targeted-Capture", "CLONEEND"]:
        return "NO"
    
    # fall back to regex
    return regex_class

final_catalog["classification_final"] = final_catalog.apply(classify_final, axis=1)

print(final_catalog["classification_final"].value_counts())
print(f"\nYES studies: {(final_catalog['classification_final']=='YES').sum()}")
print(f"YES samples: {final_catalog[final_catalog['classification_final']=='YES']['n_samples'].sum():,}")
print(f"NO studies: {(final_catalog['classification_final']=='NO').sum()}")
print(f"MAYBE studies: {(final_catalog['classification_final']=='MAYBE').sum()}")
print(f"UNCLEAR studies: {(final_catalog['classification_final']=='UNCLEAR').sum()}")

classification_final
YES        900
MAYBE      389
NO         245
UNCLEAR      1
Name: count, dtype: int64

YES studies: 900
YES samples: 98,657
NO studies: 245
MAYBE studies: 389
UNCLEAR studies: 1


In [18]:
maybe_studies = final_catalog[final_catalog["classification_final"] == "MAYBE"]

print("MAYBE studies breakdown:")
print(f"\nBy library_source:")
print(maybe_studies["library_source"].value_counts())
print(f"\nBy library_strategy:")
print(maybe_studies["library_strategy"].value_counts())
print(f"\nSample titles:")
for title in maybe_studies["paper_title"].dropna().head(5).tolist():
    print(f"  - {title}")

    

MAYBE studies breakdown:

By library_source:
library_source
METAGENOMIC           385
METATRANSCRIPTOMIC      4
Name: count, dtype: int64

By library_strategy:
library_strategy
AMPLICON    352
OTHER        35
WGS           2
Name: count, dtype: int64

Sample titles:
  - The gut microbiome stability is altered by probiotic ingestion and improved by the continuous supplementation of galactooligosaccharide.
  - Genotypic characterization of gut isolates from Atlantic salmon fry reveals beneficial microbes with biotechnological potential.
  - The role of the gut microbiota in the dietary niche expansion of fishing bats.
  - Microbial Dysregulation of the Gut-Lung Axis in Bronchiectasis.
  - Methanogenic patterns in the gut microbiome are associated with survival in a population of feral horses.


In [19]:
def classify_final(row):
    lib_source = str(row.get("library_source") or "")
    lib_strategy = str(row.get("library_strategy") or "")
    regex_class = str(row.get("classification") or "")
    
    # definite NO by library_source
    if lib_source in ["GENOMIC", "TRANSCRIPTOMIC", "VIRAL RNA",
                      "TRANSCRIPTOMIC SINGLE CELL", "GENOMIC SINGLE CELL", "SYNTHETIC"]:
        return "NO"
    
    # AMPLICON = 16S = not WGS shotgun → NO for your purposes
    if lib_strategy == "AMPLICON":
        return "NO_16S"
    
    # definite NO by library_strategy
    if lib_strategy in ["RNA-Seq", "ChIP-Seq", "Hi-C",
                        "RAD-Seq", "Targeted-Capture", "CLONEEND"]:
        return "NO"
    
    # strong YES - source METAGENOMIC + strategy WGS or METAGENOMIC
    if lib_source == "METAGENOMIC" and lib_strategy in ["WGS", "METAGENOMIC"]:
        return "YES"
    
    # source METAGENOMIC but strategy OTHER - ambiguous
    if lib_source == "METAGENOMIC" and lib_strategy == "OTHER":
        return "MAYBE"
    
    # source METAGENOMIC with no strategy info
    if lib_source == "METAGENOMIC":
        return "YES"
    
    # METATRANSCRIPTOMIC
    if lib_source == "METATRANSCRIPTOMIC":
        return "MAYBE"
    
    # fall back to regex
    return regex_class

final_catalog["classification_final"] = final_catalog.apply(classify_final, axis=1)

print(final_catalog["classification_final"].value_counts())
print(f"\nYES studies: {(final_catalog['classification_final']=='YES').sum()}")
print(f"YES samples: {final_catalog[final_catalog['classification_final']=='YES']['n_samples'].sum():,}")

classification_final
YES       867
NO_16S    363
NO        268
MAYBE      37
Name: count, dtype: int64

YES studies: 867
YES samples: 96,872


In [20]:
# save final classified catalog
final_catalog.to_csv("../results/final_catalog.tsv", sep="\t", index=False)

# save clean YES-only catalog
yes_catalog = final_catalog[final_catalog["classification_final"] == "YES"].copy()
yes_catalog.to_csv("../results/yes_catalog.tsv", sep="\t", index=False)

print(f"Full catalog saved: {len(final_catalog)} studies")
print(f"YES catalog saved: {len(yes_catalog)} studies, {yes_catalog['n_samples'].sum():,} samples")

Full catalog saved: 1535 studies
YES catalog saved: 867 studies, 96,872 samples


In [22]:
import importlib
import src.ena_fetcher
importlib.reload(src.ena_fetcher)

from src.ena_fetcher import get_taxonomy
tax = get_taxonomy("8404")
print(tax)

Error fetching taxonomy for tax_id 8404: 0 
{'kingdom': None, 'phylum': None, 'class': None, 'order': None, 'family': None, 'genus': None, 'species': None}


In [23]:
# test directly without the function
from Bio import Entrez

handle = Entrez.efetch(
    db="taxonomy",
    id="8404",
    retmode="xml"
)
records = Entrez.read(handle)
handle.close()

print(f"Records found: {len(records)}")
print(f"Scientific name: {records[0].get('ScientificName')}")
print(f"Lineage keys: {list(records[0].keys())}")

Records found: 1
Scientific name: Lithobates pipiens
Lineage keys: ['TaxId', 'ScientificName', 'OtherNames', 'ParentTaxId', 'Rank', 'Division', 'GeneticCode', 'MitoGeneticCode', 'Lineage', 'LineageEx', 'CreateDate', 'UpdateDate', 'PubDate']


In [24]:
# debug the function
tax_id = "8404"
print(f"Input: {tax_id}, type: {type(tax_id)}")
print(f"Converted: {str(int(float(tax_id)))}")

Input: 8404, type: <class 'str'>
Converted: 8404


In [25]:
def get_taxonomy(tax_id):
    if tax_id is None or str(tax_id) == "nan":
        return {"kingdom": None, "phylum": None, "class": None, 
                "order": None, "family": None, "genus": None, "species": None}
    
    try:
        converted_id = str(int(float(tax_id)))
        print(f"Fetching taxonomy for converted id: {converted_id}")
        
        handle = Entrez.efetch(
            db="taxonomy",
            id=converted_id,
            retmode="xml"
        )
        records = Entrez.read(handle)
        handle.close()
        time.sleep(0.3)
        
        taxonomy = {"kingdom": None, "phylum": None, "class": None,
                   "order": None, "family": None, "genus": None, "species": None}
        
        if records:
            taxonomy["species"] = records[0].get("ScientificName")
            lineage = records[0].get("LineageEx", [])
            for node in lineage:
                rank = node.get("Rank")
                name = node.get("ScientificName")
                if rank in taxonomy:
                    taxonomy[rank] = name
        
        return taxonomy
        
    except Exception as e:
        print(f"Error fetching taxonomy for {tax_id}: {e}")
        print(f"Error type: {type(e)}")
        return {"kingdom": None, "phylum": None, "class": None,
                "order": None, "family": None, "genus": None, "species": None}

In [26]:
get_taxonomy(8404)

Fetching taxonomy for converted id: 8404


{'kingdom': 'Metazoa',
 'phylum': 'Chordata',
 'class': 'Amphibia',
 'order': 'Anura',
 'family': 'Ranidae',
 'genus': 'Lithobates',
 'species': 'Lithobates pipiens'}

In [28]:
import importlib
import src.ena_fetcher
importlib.reload(src.ena_fetcher)

from src.ena_fetcher import get_taxonomy
tax = get_taxonomy("8404")
print(tax)

Error fetching taxonomy for tax_id 8404: 0 
{'kingdom': None, 'phylum': None, 'class': None, 'order': None, 'family': None, 'genus': None, 'species': None}


In [29]:
import importlib
import src.ena_fetcher
importlib.reload(src.ena_fetcher)

# check if function exists in module
print(dir(src.ena_fetcher))
print("\nget_taxonomy in module:", hasattr(src.ena_fetcher, 'get_taxonomy'))

['ENA_PORTAL_URL', 'Entrez', 'GENERIC_SCIENTIFIC_NAMES', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'fetch_abstract_from_pmid', 'fetch_pubmed_abstract', 'fetch_pubmed_abstract_by_title', 'fetch_pubmed_id', 'fetch_runs_for_study', 'fetch_study_origin', 'get_taxonomy', 'pd', 'requests', 'resolve_host_species', 'search_ena_studies', 'time']

get_taxonomy in module: True


In [30]:
result = src.ena_fetcher.get_taxonomy("8404")
print(result)

Error fetching taxonomy for tax_id 8404: 0 
{'kingdom': None, 'phylum': None, 'class': None, 'order': None, 'family': None, 'genus': None, 'species': None}


In [32]:
from src.fetcher import configure_entrez
configure_entrez()

result = src.ena_fetcher.get_taxonomy("8404")
print(result)

Entrez configured with email: akharya@ucsd.edu
Error fetching taxonomy for tax_id 8404: 0 
{'kingdom': None, 'phylum': None, 'class': None, 'order': None, 'family': None, 'genus': None, 'species': None}


In [33]:
import importlib
import src.ena_fetcher
importlib.reload(src.ena_fetcher)

from src.ena_fetcher import get_taxonomy
tax = get_taxonomy("8404")
print(tax)

{'kingdom': 'Metazoa', 'phylum': 'Chordata', 'class': 'Amphibia', 'order': 'Anura', 'family': 'Ranidae', 'genus': 'Lithobates', 'species': 'Lithobates pipiens'}


In [ ]:
from concurrent.futures import ThreadPoolExecutor

def get_taxonomy_safe(row):
    tax_id = row.get("host_tax_id")
    if pd.isna(tax_id) if pd.notna(tax_id) else True:
        return row["study_accession"], {"kingdom": None, "phylum": None, "class": None,
                                        "order": None, "family": None, "genus": None, "species": None}
    try:
        taxonomy = get_taxonomy(str(tax_id))
        return row["study_accession"], taxonomy
    except Exception as e:
        return row["study_accession"], {"kingdom": None, "phylum": None, "class": None,
                                        "order": None, "family": None, "genus": None, "species": None}

yes_catalog = final_catalog[final_catalog["classification_final"] == "YES"].copy()
rows = yes_catalog.to_dict("records")

print(f"Fetching taxonomy for {len(yes_catalog)} studies...")

with ThreadPoolExecutor(max_workers=3) as executor:
    results = list(executor.map(get_taxonomy_safe, rows))

taxonomy_data = {acc: tax for acc, tax in results}

yes_catalog["host_kingdom"] = yes_catalog["study_accession"].map(lambda x: taxonomy_data.get(x, {}).get("kingdom"))
yes_catalog["host_phylum"] = yes_catalog["study_accession"].map(lambda x: taxonomy_data.get(x, {}).get("phylum"))
yes_catalog["host_class"] = yes_catalog["study_accession"].map(lambda x: taxonomy_data.get(x, {}).get("class"))
yes_catalog["host_order"] = yes_catalog["study_accession"].map(lambda x: taxonomy_data.get(x, {}).get("order"))
yes_catalog["host_family"] = yes_catalog["study_accession"].map(lambda x: taxonomy_data.get(x, {}).get("family"))
yes_catalog["host_genus"] = yes_catalog["study_accession"].map(lambda x: taxonomy_data.get(x, {}).get("genus"))
yes_catalog["host_species_taxonomy"] = yes_catalog["study_accession"].map(lambda x: taxonomy_data.get(x, {}).get("species"))

print(f"\nHost class populated: {yes_catalog['host_class'].notna().sum()}/{len(yes_catalog)}")
print(f"\nTop host classes:")
print(yes_catalog["host_class"].value_counts().head(10))                                                                                                                                                                                                                                                                                                                                

Fetching taxonomy for 867 studies...

Host class populated: 464/867

Top host classes:
host_class
Mammalia          346
Aves               57
Actinopteri        32
Insecta            19
Chondrichthyes      3
Magnoliopsida       2
Malacostraca        1
Amphibia            1
Lepidosauria        1
Arachnida           1
Name: count, dtype: int64


In [35]:
yes_catalog.to_csv("../results/yes_catalog.tsv", sep="\t", index=False)
print(f"Saved {len(yes_catalog)} studies")
print(f"Columns: {list(yes_catalog.columns)}")

Saved 867 studies
Columns: ['study_accession', 'host_species', 'host_tax_id', 'body_site', 'country', 'n_samples', 'library_strategy', 'data_source', 'pmid', 'pmid_str', 'paper_title', 'first_author', 'last_author', 'host_class', 'abstract', 'pub_year', 'journal', 'classification', 'needs_review', 'library_source', 'classification_final', 'host_kingdom', 'host_phylum', 'host_order', 'host_family', 'host_genus', 'host_species_taxonomy']
